In [1]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

$$
d^{\ell}_{m-1,n}(\beta) = \lambda _{m}a_{m-1}d^{\ell}_{m,n}(\beta) - \frac{a_{m-1}}{a_m}d^{\ell}_{m+1,n}(\beta) \\
 \lambda _{m} = \frac{n-m\cos{\beta}}{\sin{\beta}} \\
 a_{m} = \frac{2}{\sqrt{(\ell -m)(\ell+m+1)}}
$$

In [2]:
l = 100
n = 29
beta = pi/7

d_rec = wignerd_mdown_from_l(l, n, beta)
d_ref = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in 0:l]

maximum(abs.(d_rec .- d_ref))

11340.873001706941

In [3]:
d_ref

101-element Vector{ComplexF64}:
   -0.014588311057730306 + 0.0im
    -0.11332703283714206 + 0.0im
    -0.13148117688616207 + 0.0im
    -0.05071951002184934 + 0.0im
     0.07030822718329764 - 0.0im
     0.13271812964999272 - 0.0im
     0.07898868637458265 - 0.0im
   -0.047138729305299935 + 0.0im
    -0.12824509032104955 + 0.0im
    -0.08146848715837743 + 0.0im
                         ⋮
  -5.977414751743095e-13 + 0.0im
   9.323750451718662e-14 - 0.0im
 -1.3094891003068569e-14 + 0.0im
  1.4206515062771451e-15 - 0.0im
  2.7008750293912423e-16 - 0.0im
  -2.812227943981304e-16 + 0.0im
   1.121248709190417e-15 - 0.0im
  1.3712990707366896e-15 - 0.0im
  4.2638081341043496e-16 - 0.0im

In [4]:
d_rec

101-element Vector{ComplexF64}:
       827.5225049687124 + 0.0im
       6428.480289661222 + 0.0im
       7458.274807995467 + 0.0im
       2877.066153715339 + 0.0im
     -3988.2368869423863 + 0.0im
      -7528.441001025249 + 0.0im
      -4480.636267914229 + 0.0im
      2673.9462300603327 + 0.0im
       7274.707673317511 + 0.0im
       4621.303062604307 + 0.0im
                         ⋮
   3.3851823068119175e-8 + 0.0im
   -5.381557215671708e-9 + 0.0im
   7.864712239066948e-10 - 0.0im
 -1.0462478235841119e-10 + 0.0im
  1.2497979612821988e-11 + 0.0im
  -1.314067378208217e-12 - 0.0im
  1.1779518718868942e-13 + 0.0im
  -8.490990571621479e-15 + 0.0im
  4.2638081341043496e-16 - 0.0im

In [5]:
# ComplexF64 に ldexp をかける（2^e 倍）
@inline function cldexp(z::ComplexF64, e::Int)
    return ComplexF64(ldexp(real(z), e), ldexp(imag(z), e))
end

function wignerd_mdown_from_l_scaled(l::Int, n::Int, beta::Real;
    eps_sin::Real=1e-14,   # sinβ がこれ以下なら direct に逃げる
    beta_switch::Real=0.0, # 0 なら自動で決める
)
    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(n) <= l) || throw(ArgumentError("n must satisfy |n| <= l"))

    dvals = zeros(ComplexF64, l + 1)

    # ---- 小β領域の分岐（sinβ判定だけだと足りないことがある）----
    β = float(beta)
    if beta_switch == 0.0
        # 雑だけど実務で効く：lが大きいほど小βを厳しく
        # （必要ならここは調整してOK）
        beta_switch = 1e-6 * (l + 1)
    end

    if abs(β) < beta_switch || abs(sin(β)) <= eps_sin
        for m in 0:l
            dvals[m + 1] = WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0)
        end
        return dvals
    end

    # ---- 半角で sinβ, cosβ, λm を計算（キャンセル低減）----
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    # seed: d_{l,n}
    d_l_true = WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0)
    dvals[l + 1] = d_l_true
    l == 0 && return dvals

    # ---- スケーリング状態：内部値 d_internal と指数 e（真値 = d_internal * 2^e）----
    e = 0

    # 内部状態（最初は真値をそのまま）
    d_m_plus1 = 0.0 + 0.0im      # d_{m+1}
    d_m       = d_l_true         # d_{m}

    # リスケ条件：conviqtっぽく「安全帯」に戻す（2の冪だけ使う）
    # ※値は適当に安全側。必要なら調整。
    BIG  = 2.0^450
    SMALL = 2.0^-450
    SHIFT = 200  # 一度に動かす2冪の指数

    @inline function rescale!(x::ComplexF64, y::ComplexF64, e::Int)
        # x,y を同じ2冪でまとめてスケール（線形同次なのでOK）
        ax = abs(x); ay = abs(y)
        amax = max(ax, ay)
        if amax == 0.0
            return x, y, e
        elseif amax > BIG
            # 2^SHIFT で割る
            x = cldexp(x, -SHIFT)
            y = cldexp(y, -SHIFT)
            e += SHIFT
        elseif amax < SMALL
            # 2^SHIFT で掛ける
            x = cldexp(x, SHIFT)
            y = cldexp(y, SHIFT)
            e -= SHIFT
        end
        return x, y, e
    end

    # 初回も念のため
    d_m, d_m_plus1, e = rescale!(d_m, d_m_plus1, e)

    for m in l:-1:1
        # a_{m-1}
        am1 = 2 / sqrt((l - (m - 1)) * (l + (m - 1) + 1))

        # λm を半角形で（n-m + 2m sh^2）/sinβ
        λm = ((n - m) + 2m * (sh*sh)) / sinβ

        d_m_minus1 = if m == l
            (λm * am1) * d_m
        else
            am = 2 / sqrt((l - m) * (l + m + 1))
            (λm * am1) * d_m - (am1 / am) * d_m_plus1
        end

        # 次ステップに備えてリスケ
        d_m_minus1, d_m, e = rescale!(d_m_minus1, d_m, e)

        # ここで「真値」を保存（ldexpで指数を反映）
        # dvals の添字は m-1 -> (m) だが、あなたの元コードに合わせて m を保存しているので踏襲
        dvals[m] = cldexp(d_m_minus1, e)

        # 状態更新（内部値のまま）
        d_m_plus1 = d_m
        d_m = d_m_minus1
    end

    # 最後に m=0 の要素を保存（ループ内で dvals[1] を入れていないため）
    dvals[1] = cldexp(d_m, e)

    return dvals
end

function wignerd_mdown_ratio_norm(l::Int, n::Int, beta::Real;
        eps_sin::Real=1e-14)

    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(n) <= l) || throw(ArgumentError("n must satisfy |n| <= l"))

    β = float(beta)
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    dvals = zeros(Float64, l+1)

    # まず危険域は安全側の実装へ逃がす（β→0 の特異性を踏むので大事）
    if abs(sinβ) <= eps_sin
        for m in 0:l
            dvals[m+1] = real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0))
        end
        return dvals
    end

    # --- 1) ratio r_m = d_{m+1}/d_m を下向きに計算 ---
    r = zeros(Float64, l+1)   # r[ m+1 ] = r_m
    r[l+1] = 0.0              # r_l = 0 （d_{l+1}=0）

    for m in l:-1:1
        am1 = 2 / sqrt((l-(m-1))*(l+(m-1)+1))
        # λm は半角で（キャンセルが減る）
        λm = ((n - m) + 2m*sh*sh) / sinβ
        A  = λm * am1

        C = if m == l
            0.0                 # d_{l+1}項は無いので C=0
        else
            am = 2 / sqrt((l-m)*(l+m+1))
            -(am1 / am)
        end

        denom = A + C*r[m+1]    # A + C r_m
        # denom が 0 に近いときはここが爆心地
        r[m] = 1.0 / denom      # r_{m-1}
    end

    # --- 2) 復元：d_l を種にして下へ ---
    #d_l = real(WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0))
    d_l = seed_d_l_n_conviqt(l, n, beta)
    dvals[l+1] = d_l
    for m in l:-1:1
        # r[m] = d_m / d_{m-1} なので d_{m-1} = d_m / r[m]
        dvals[m] = dvals[m+1] / r[m]
    end

    # --- 3) 正規化：列（固定n）のノルムは 1 になるべき ---
    # 直交性：∑_m d_{m n}^2 = 1
    norm = sqrt(sum(abs2, dvals))
    dvals ./= norm

    # dvals[m+1] = d^l_{m,0}  (m=0..l) だと仮定
    #norm2_full = dvals[1]^2 + 2*sum(dvals[2:end].^2)
    #dvals ./= sqrt(norm2_full)

    # 位相（符号）を seed に合わせたいなら：
    # （正規化で全体符号が反転しても同じ解なので、好みで固定）
    if d_l != 0 && sign(dvals[l+1]) != sign(d_l)
        dvals .*= -1
    end

    return dvals
end

using SpecialFunctions

"""
conviqt式(27)(31)に対応する seed: d^l_{l,n}(β)
- logで絶対値を計算し、符号は (-1)^(l-n) として分離
- exp のオーバーフロー/アンダーフローを避けるため 2冪スケーリングで復元
"""
function seed_d_l_n_conviqt(l::Int, n::Int, beta::Real)
    (abs(n) <= l) || throw(ArgumentError("|n| must be <= l"))
    β = float(beta)

    sh = sin(β/2)
    ch = cos(β/2)

    # β→0 のとき sin(β/2)=0 で logが -Inf になるので、数学的にも多くが0
    # (l-n)>0 なら 0, (l-n)=0 なら (cos)^(2l)*A に落ちる
    if sh == 0.0
        if l == n
            # d^l_{l,l}(β)= (cos(β/2))^(2l) * A だが A=1
            return ch^(2l)
        else
            return 0.0
        end
    end

    # A = sqrt( (2l)! / ((l+n)!(l-n)!) )
    # logA = 0.5*(ln(2l)! - ln(l+n)! - ln(l-n)!)
    logA = 0.5 * (loggamma(2l + 1) - loggamma(l + n + 1) - loggamma(l - n + 1))

    # ln|d| = logA + (l+n)ln|cos| + (l-n)ln|sin|
    # (31)の形そのもの :contentReference[oaicite:2]{index=2}
    logabs = logA + (l + n) * log(abs(ch)) + (l - n) * log(abs(sh))

    # 符号：β∈(0,π) なら sin(β/2)>0 なので (-sin)^(l-n)=(-1)^(l-n)*sin^(l-n)
    sgn = isodd(l - n) ? -1.0 : 1.0

    # exp(logabs) を直接やると under/overflow するので 2冪で復元
    log2abs = logabs / log(2.0)
    e2 = floor(Int, log2abs)               # 2^e2 の指数
    mant = exp(logabs - e2 * log(2.0))     # mant ∈ (0,2) くらいに収まる
    return sgn * ldexp(mant, e2)
end

using SpecialFunctions

# d^l_{l,n}(β) を log で作る（倍率は正規化で消えるが、極小βでの安定性のため）
function seed_d_l_n_conviqt(l::Int, n::Int, beta::Real)
    (abs(n) <= l) || throw(ArgumentError("|n| <= l"))
    β = float(beta)
    sh = sin(β/2); ch = cos(β/2)

    if sh == 0.0
        return (l == n) ? ch^(2l) : 0.0
    end

    logA = 0.5 * (loggamma(2l + 1) - loggamma(l + n + 1) - loggamma(l - n + 1))
    logabs = logA + (l + n) * log(abs(ch)) + (l - n) * log(abs(sh))
    sgn = isodd(l - n) ? -1.0 : 1.0

    # 2冪で復元（overflow/underflow回避）
    log2abs = logabs / log(2.0)
    e2 = floor(Int, log2abs)
    mant = exp(logabs - e2 * log(2.0))
    return sgn * ldexp(mant, e2)
end

"""
conviqt式(7)の m0 方向再帰（ratio）で、固定 n に対し d^l_{m,n}(β) を m=0..l で返す。
- まず d^l_{n,m0}(β) を m0=l..0 で作り、
- 対称性 d_{mn}=(-1)^(m-n)d_{nm} で d^l_{m,n} に変換する
"""
function wignerd_mvec_fixedn_conviqt7(l::Int, n::Int, beta::Real; eps_sin::Real=1e-14)
    (l >= 0) || throw(ArgumentError("l>=0"))
    (abs(n) <= l) || throw(ArgumentError("|n|<=l"))

    β = float(beta)
    sh = sin(β/2); ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    # 返すのは m=0..l
    out = zeros(Float64, l+1)

    # βが極小なら別ルート（ここは好みで：あなたのWignerD.jl呼び出しでもOK）
    if abs(sinβ) <= eps_sin
        # β≈0 の極限は δ_{mn} なので m=|n|以外は0に近い
        # ただし厳密にやりたいなら外部関数でOK
        for m in 0:l
            out[m+1] = (m == n) ? 1.0 : 0.0
        end
        return out
    end

    mfix = n  # 式(7)の "m" を固定 = n
    # r[m0] = d_{mfix,m0+1} / d_{mfix,m0}
    r = zeros(Float64, l+1)
    r[l+1] = 0.0  # 境界：d_{mfix,l+1}=0 とみなせるので r_l=0

    # m0 = l,l-1,...,1 で r_{m0-1} を作る
    for m0 in l:-1:1
        t = (-mfix + m0*cosβ) / sinβ

        # α0 = 0.5*sqrt((l+m0)(l-m0+1))
        α0 = 0.5 * sqrt((l + m0) * (l - m0 + 1))
        # β0 = 0.5*sqrt((l-m0)(l+m0+1))
        β0 = 0.5 * sqrt((l - m0) * (l + m0 + 1))

        # 式(7)より：
        # α0 * d_{m0-1} = t*d_{m0} - β0*d_{m0+1}
        # なので q = d_{m0-1}/d_{m0} = (t - β0*r_m0)/α0
        q = (t - β0*r[m0+1]) / α0

        # r_{m0-1} = d_{m0}/d_{m0-1} = 1/q
        r[m0] = 1.0 / q
    end

    # 復元：seed として d_{mfix,l} を与えて、m0を下げる
    d_nm0 = zeros(Float64, l+1)   # d_nm0[m0+1] = d^l_{n,m0}
    d_nm0[l+1] = seed_d_l_n_conviqt(l, mfix, β)  # まず d^l_{l,mfix}
    # 欲しいのは d^l_{mfix,l} だが、d_{mfix,l} = (-1)^(mfix-l) d_{l,mfix}
    if isodd(mfix - l)
        d_nm0[l+1] *= -1
    end

    for m0 in l:-1:1
        # r[m0] = d_{m0}/d_{m0-1} なので d_{m0-1} = d_{m0}/r[m0]
        d_nm0[m0] = d_nm0[m0+1] / r[m0]
    end

    # ここで d_nm0[m+1] = d^l_{n,m}
    # 対称性で d^l_{m,n} に戻す：d_{m,n} = (-1)^(m-n) d_{n,m}
    for m in 0:l
        v = d_nm0[m+1]
        if isodd(m - n)
            v = -v
        end
        out[m+1] = v
    end

    # 正規化：もし con**viqt 側も「m=-l..l を含む列ノルム1」を使っているなら、
    # m>=0 しか持ってない out だけで norm を取ると一致しません。
    # まずは比較条件に合わせて、ここは "しない" か、同じ範囲で行ってください。
    # （とりあえず m=0..l だけでの正規化を入れるなら↓）
    # out ./= sqrt(sum(abs2,out))

    return out
end

wignerd_mvec_fixedn_conviqt7

In [30]:
l = 100
n = 20
beta = pi/7

d_rec = wignerd_mvec_fixedn_conviqt7(l, n, beta)
d_rec2 = wignerd_mdown_from_l_scaled(l, n, beta)
d_rec3 = wignerd_mdown_ratio_norm(l, n, beta)
d_ref = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in 0:l]

101-element Vector{ComplexF64}:
   -0.01802862242359784 + 0.0im
    0.10474869743828857 - 0.0im
    0.10980002194389303 - 0.0im
  -0.013094106752379195 + 0.0im
   -0.12022715751886555 + 0.0im
   -0.07740519185477962 + 0.0im
    0.06519237964532873 - 0.0im
    0.12118115834395032 - 0.0im
   0.011083172744630766 - 0.0im
   -0.11475159272202033 + 0.0im
                        ⋮
 4.0246927888383595e-15 - 0.0im
  9.479115624771587e-16 - 0.0im
 4.1141477263211063e-16 - 0.0im
  5.535938261067057e-16 - 0.0im
  6.227392240374654e-16 - 0.0im
  5.176073643879524e-16 - 0.0im
  8.367745554686574e-16 - 0.0im
 -7.457277288951977e-16 + 0.0im
 -8.793051261340683e-16 + 0.0im

In [31]:
maximum(abs.(d_rec3 .- d_ref))

0.05034122485797099

In [32]:
maximum(abs.(d_rec2 .- d_ref))

1.4793996884285955e9

In [35]:
l = 100
n = 20
beta = pi/7

0.4487989505128276

In [36]:
# diagnostics for ratio method
norm_half_ref = sqrt(sum(abs2, d_ref))
d_ref_full = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in -l:l]
norm_full_ref = sqrt(sum(abs2, d_ref_full))

function ratio_denominators(l, n, beta)
    β = float(beta)
    sh = sin(β/2)
    sinβ = 2sh*cos(β/2)
    r = zeros(Float64, l+1)
    r[l+1] = 0.0
    denoms = zeros(Float64, l)
    for m in l:-1:1
        am1 = 2 / sqrt((l-(m-1))*(l+(m-1)+1))
        λm = ((n - m) + 2m*sh*sh) / sinβ
        A = λm * am1
        C = (m == l) ? 0.0 : -(am1 / (2 / sqrt((l-m)*(l+m+1))))
        denom = A + C*r[m+1]
        denoms[m] = denom
        r[m] = 1.0 / denom
    end
    return minimum(abs.(denoms)), maximum(abs.(r))
end

min_abs_denom, max_abs_ratio = ratio_denominators(l, n, 1.5)

(
    norm_half_ref = norm_half_ref,
    norm_full_ref = norm_full_ref,
    max_err_ratio = maximum(abs.(d_rec3 .- d_ref)),
    min_abs_denom = min_abs_denom,
    max_abs_ratio = max_abs_ratio,
    seed_diff = seed_d_l_n_conviqt(l, n, beta) - d_ref[end],
)

(norm_half_ref = 0.7987874363414407, norm_full_ref = 0.9999999999999968, max_err_ratio = 0.05034122485797099, min_abs_denom = 0.0068717821173669424, max_abs_ratio = 145.52265815773123, seed_diff = 8.793051262528512e-16 - 0.0im)

In [37]:
l

100

In [17]:
n

2

In [18]:
beta

0.4487989505128276

In [38]:
using SpecialFunctions

# -----------------------------
# seed: d^l_{l,n}(β)  (conviqt式(27)(31)相当)
# -----------------------------
function seed_d_l_n_conviqt(l::Int, n::Int, beta::Real)
    (abs(n) <= l) || throw(ArgumentError("|n| <= l"))
    β = float(beta)

    sh = sin(β/2)
    ch = cos(β/2)

    # β=0 の極限
    if sh == 0.0
        return (l == n) ? ch^(2l) : 0.0
    end

    # A = sqrt((2l)! / ((l+n)!(l-n)!))
    logA = 0.5 * (loggamma(2l + 1) - loggamma(l + n + 1) - loggamma(l - n + 1))

    # ln|d^l_{l,n}| = logA + (l+n)ln|cos(β/2)| + (l-n)ln|sin(β/2)|
    logabs = logA + (l + n) * log(abs(ch)) + (l - n) * log(abs(sh))

    # (-sin(β/2))^(l-n) の符号
    sgn = isodd(l - n) ? -1.0 : 1.0

    # 2冪で復元（overflow/underflow回避）
    log2abs = logabs / log(2.0)
    e2 = floor(Int, log2abs)
    mant = exp(logabs - e2 * log(2.0))
    return sgn * ldexp(mant, e2)
end

# -----------------------------
# 固定 m で d^l_{m,n}(β) を n=0..l で返す（conviqt式(7)の方向）
#
# まず ratio で d_{m,n+1}/d_{m,n} を n=l→0 に向けて作り、
# seed として d_{m,l} を与えて復元する。
#
# 注意:
# - 返すのは n=0..l のみ
# - 比較時の正規化は相手(conviqt)の規約に合わせて行うこと
# -----------------------------
function wignerd_n_down_fixedm_conviqt7_ratio(l::Int, m::Int, beta::Real;
    eps_sin::Real = 1e-14,
    normalize::Bool = false,   # 比較相手に合わせる。基本 false 推奨
)
    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(m) <= l) || throw(ArgumentError("|m| must be <= l"))

    β = float(beta)
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2 * sh * ch
    cosβ = ch*ch - sh*sh

    # 出力: dvals[n+1] = d^l_{m,n}(β), n=0..l
    dvals = zeros(Float64, l + 1)

    # βが極小なら安全側へ（必要なら WignerD.jl に逃がしてもOK）
    if abs(sinβ) <= eps_sin
        for n in 0:l
            dvals[n+1] = (m == n) ? 1.0 : 0.0
        end
        return dvals
    end

    # ratio 配列: r[n+1] = d_{m,n+1} / d_{m,n}
    # 実際には n=l..1 で降ろしながら r[n] (= d_{m,n}/d_{m,n-1}) を埋める
    r = zeros(Float64, l + 1)
    r[l+1] = 0.0   # 境界: d_{m,l+1}=0 とみなす => r_l = d_{m,l+1}/d_{m,l}=0

    # conviqt式(7):
    # ((-m + n cosβ)/sinβ) d_{m,n}
    # = (1/2)√((l+n)(l-n+1)) d_{m,n-1}
    # + (1/2)√((l-n)(l+n+1)) d_{m,n+1}
    #
    # これを ratio r_n = d_{m,n+1}/d_{m,n} を使って解く:
    # α_n d_{m,n-1} = t_n d_{m,n} - β_n d_{m,n+1}
    # q_n = d_{m,n-1}/d_{m,n} = (t_n - β_n r_n)/α_n
    # r_{n-1} = d_{m,n}/d_{m,n-1} = 1/q_n
    for n in l:-1:1
        t = (-m + n * cosβ) / sinβ
        α = 0.5 * sqrt((l + n) * (l - n + 1))   # coeff of d_{m,n-1}
        βc = 0.5 * sqrt((l - n) * (l + n + 1))  # coeff of d_{m,n+1}

        q = (t - βc * r[n+1]) / α
        r[n] = 1.0 / q
    end

    # seed: d_{m,l} を対称性から作る
    # まず d_{l,m} を閉形式で作って、d_{m,l} = (-1)^(m-l) d_{l,m}
    d_lm = seed_d_l_n_conviqt(l, m, β)   # d^l_{l,m}
    d_ml = isodd(m - l) ? -d_lm : d_lm   # d^l_{m,l}

    dvals[l+1] = d_ml

    # 復元: r[n] = d_{m,n}/d_{m,n-1} なので d_{m,n-1} = d_{m,n} / r[n]
    for n in l:-1:1
        dvals[n] = dvals[n+1] / r[n]
    end

    # 任意の正規化（比較相手に合わせる）
    if normalize
        nrm = sqrt(sum(abs2, dvals))
        if nrm != 0.0
            dvals ./= nrm
        end
    end

    return dvals
end

wignerd_n_down_fixedm_conviqt7_ratio (generic function with 1 method)

In [58]:
l = 100
m = 20
beta = 1e-4
d_m_fixed = wignerd_n_down_fixedm_conviqt7_ratio(l, m, beta; normalize=false)
# d_m_fixed[n+1] = d^l_{m,n}(beta), n=0..l

101-element Vector{Float64}:
  1.6516405781017348e48
 -4.368100940510082e44
  1.2192924454686498e41
 -3.6029700772477384e37
  1.1308714971481863e34
 -3.7846203483310765e30
  1.3563728320606762e27
 -5.231926304742714e23
  2.1847603661527063e20
 -9.944624444202187e16
  ⋮
  3.356318940787027e-278
  9.033041767087224e-283
  2.2491695236846635e-287
  5.128895661713746e-292
  1.0563146821569253e-296
  1.9254653441522594e-301
  3.008183553348234e-306
  3.7982958029797e-311
  3.3572509e-316

In [59]:
d_ref_full = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for n in 0:l]


101-element Vector{ComplexF64}:
   2.374836438612249e-15 - 0.0im
   2.836543933765201e-17 - 0.0im
  -3.238728729648699e-15 + 0.0im
  -3.781155076543197e-18 + 0.0im
 -2.5379004453540688e-15 + 0.0im
   4.052205619664573e-18 - 0.0im
 -3.3089850304257595e-15 + 0.0im
 -2.6095391039010485e-17 + 0.0im
   1.883042333172824e-15 - 0.0im
  1.2122735541103546e-17 - 0.0im
                         ⋮
 -2.8371446233005113e-15 + 0.0im
 -1.2768964438649819e-17 + 0.0im
  4.2134928948318315e-15 - 0.0im
  6.8093485954028825e-18 - 0.0im
   7.194296555713354e-16 - 0.0im
  1.4881074827347914e-19 - 0.0im
  3.3137088969127904e-17 - 0.0im
  -7.231586150858504e-19 + 0.0im
   1.718835682596729e-15 - 0.0im

In [60]:
maximum(abs.(d_m_fixed-d_ref_full))

1.6516405781017348e48

In [40]:
for n in (0, 1, 2, l)
    v1 = d_m_fixed[n+1]
    v2 = real(WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0))
    @show n v1 v2 (v1-v2) / (abs(v2)+1e-300)
end

n = 0
v1 = 5.237302419851003e-5
v2 = 5.237302646837749e-5
(v1 - v2) / (abs(v2) + 1.0e-300) = -4.334039129600329e-8
n = 1
v1 = -0.010221990884355854
v2 = -0.010221990884379058
(v1 - v2) / (abs(v2) + 1.0e-300) = 2.2699747512125957e-12
n = 2
v1 = 0.999896002699707
v2 = 0.9998960026996998
(v1 - v2) / (abs(v2) + 1.0e-300) = 7.217200229403101e-15
n = 20
v1 = 1.2844798204035657e-54
v2 = 1.9545962884250784e-16
(v1 - v2) / (abs(v2) + 1.0e-300) = -1.0


In [10]:
using WignerD

"""
固定 n に対して d^l_{m,n}(β) を m=0..mmax まで前進再帰で計算する（試作版）
- seed は m=0,1 を WignerD.wignerDjmn で直接計算
- m>=2 をあなたの m-再帰を変形して前進 (m↑)
- 必要なら direct 値との誤差を返す（検証用）

返り値:
  dvals::Vector{Float64}  # dvals[m+1] = d^l_{m,n}(β), m=0..mmax
"""
function wignerd_mup_from_smallm(
    l::Int, n::Int, beta::Real;
    mmax::Int = l,
    check_direct::Bool = false,
)
    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(n) <= l) || throw(ArgumentError("|n| must be <= l"))
    (0 <= mmax <= l) || throw(ArgumentError("mmax must satisfy 0 <= mmax <= l"))

    β = BigFloat(beta)
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    dvals = zeros(Float64, mmax + 1)

    # β≈0 は δ_{mn} で十分（必要なら direct に逃がしてもよい）
    if abs(sinβ) < 1e-15
        for m in 0:mmax
            dvals[m+1] = (m == n) ? 1.0 : 0.0
        end
        if check_direct
            dref = [real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0)) for m in 0:mmax]
            relerr = [abs(dvals[i]-dref[i]) / (abs(dref[i]) + 1e-300) for i in eachindex(dvals)]
            return dvals, dref, relerr
        end
        return dvals
    end

    # --- seed: m=0,1 を direct に計算 ---
    dvals[1] = BigFloat(real(WignerD.wignerDjmn(l, 0, n, 0.0, β, 0.0)))
    if mmax >= 1
        dvals[2] = BigFloat(real(WignerD.wignerDjmn(l, 1, n, 0.0, β, 0.0)))
    end

    # --- 前進再帰: m=1..mmax-1 から d_{m+1} を作る ---
    # 元の式:
    # d_{m-1} = (λ_m a_{m-1}) d_m - (a_{m-1}/a_m) d_{m+1}
    # => d_{m+1} = (a_m/a_{m-1}) * ((λ_m a_{m-1}) d_m - d_{m-1})
    #           = a_m * λ_m * d_m - (a_m/a_{m-1}) d_{m-1}
    for m in 1:(mmax-1)
        # a_m, a_{m-1}
        am   = 2 / sqrt((l - m) * (l + m + 1))
        am1  = 2 / sqrt((l - (m - 1)) * (l + (m - 1) + 1))

        # λ_m = (n - m cosβ)/sinβ  (半角形で計算)
        λm = ((n - m) + 2m*sh*sh) / sinβ

        d_m_minus1 = dvals[m]     # index m   -> d_{m-1}
        d_m        = dvals[m+1]   # index m+1 -> d_m

        d_mp1 = am * λm * d_m - (am / am1) * d_m_minus1
        dvals[m+2] = d_mp1
    end

    if check_direct
        dref = [real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0)) for m in 0:mmax]
        relerr = [abs(dvals[i]-dref[i]) / (abs(dref[i]) + 1e-300) for i in eachindex(dvals)]
        diff = [abs(dvals[i]-dref[i]) - (abs(dref[i]) + 1e-300) for i in eachindex(dvals)]
        return dvals, dref, relerr, diff
    end

    return dvals
end

wignerd_mup_from_smallm

In [11]:
l = 1200
n = 20
beta = 1e-4
mmax = 20

dvals, dref, relerr, diff = wignerd_mup_from_smallm(l, n, beta; mmax=mmax, check_direct=true)

@show maximum(relerr)
for m in 0:mmax
    println("m=$m  rec=$(dvals[m+1])  ref=$(dref[m+1])  relerr=$(relerr[m+1])  diff=$(diff[m+1])")
end

maximum(relerr) = 3.93986927083670607900624903938352330477270812833497631713253829698498795940967e+24
m=0  rec=-2.1273158984152747e-15  ref=-2.127315898415274727125120350883299377322823601602120890744850633619119326649397e-15  relerr=3.385432653504680655837551700683416072870994673448942406219028741921230413606183e-17  diff=-2.127315898415274655106273282736127397713637478260384483519374043680727481842041e-15
m=1  rec=-1.1303150187857456e-16  ref=-1.130315018785745651072623941561628072630263607838658687301880659059625968943473e-16  relerr=3.752661750983512723668246636485388696623455091955370603715434713901778942661202e-17  diff=-1.130315018785745608655724565966844545248499548069712195008662547479616478085518e-16
m=2  rec=-3.3651111884209825e-14  ref=-4.007201039583083613451369094006371969545919713463870441803870266443420498367259e-15  relerr=7.397660000535173253155030486232087236549654992488712703495581507087175169141988  diff=2.56367098050436575268331849447388603555759460172339819240133

In [71]:
using WignerD
using Statistics
using Base.Threads

"""
離散β平均を計算する（堅い版）
avg = mean_i d^l_{m,n}(β_i)

- β配列をそのまま回す
- WignerD.wignerDjmn を直接使う
- 精度優先
"""
function mean_wignerd_discrete(l::Int, m::Int, n::Int, betas::AbstractVector{<:Real})
    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(m) <= l && abs(n) <= l) || throw(ArgumentError("|m|,|n| <= l"))
    isempty(betas) && throw(ArgumentError("betas must be non-empty"))

    s = 0.0
    @inbounds for β in betas
        s += real(WignerD.wignerDjmn(l, m, n, 0.0, float(β), 0.0))
    end
    return s / length(betas)
end

mean_wignerd_discrete

In [72]:
using WignerD
using Statistics
using Base.Threads

"""
d^l_{m,n}(β_i) を全βについて計算して返す
"""
function wignerd_values_over_betas(l::Int, m::Int, n::Int, betas::AbstractVector{<:Real})
    vals = Vector{Float64}(undef, length(betas))
    @threads for i in eachindex(betas)
        β = float(betas[i])
        vals[i] = real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0))
    end
    return vals
end

"""
離散平均 + 標準偏差も返す（解析用）
"""
function mean_std_wignerd_discrete(l::Int, m::Int, n::Int, betas::AbstractVector{<:Real})
    vals = wignerd_values_over_betas(l, m, n, betas)
    return (mean = mean(vals), std = std(vals), vals = vals)
end

mean_std_wignerd_discrete

In [73]:
using WignerD
using Statistics

# ---------------------------
# Direct（基準）
# ---------------------------
function mean_wignerd_direct(l::Int, m::Int, n::Int, betas::AbstractVector{<:Real})
    N = length(betas)
    N == 0 && throw(ArgumentError("betas is empty"))
    s = 0.0
    @inbounds for β in betas
        s += real(WignerD.wignerDjmn(l, m, n, 0.0, float(β), 0.0))
    end
    return s / N
end

# ---------------------------
# Binned（近似）
# - 各ビンで β の平均値を代表点に使う
# ---------------------------
function mean_wignerd_binned(
    l::Int, m::Int, n::Int, betas::AbstractVector{<:Real};
    nbins::Int = 2000,
)
    N = length(betas)
    N == 0 && throw(ArgumentError("betas is empty"))
    nbins >= 1 || throw(ArgumentError("nbins must be >= 1"))

    βmin = minimum(betas)
    βmax = maximum(betas)

    if βmax == βmin
        return real(WignerD.wignerDjmn(l, m, n, 0.0, float(βmin), 0.0))
    end

    counts = zeros(Int, nbins)
    sumsβ  = zeros(Float64, nbins)

    invw = nbins / (βmax - βmin)

    @inbounds for βraw in betas
        β = float(βraw)
        k = clamp(Int(floor((β - βmin) * invw)) + 1, 1, nbins)
        counts[k] += 1
        sumsβ[k] += β
    end

    total = 0.0
    @inbounds for k in 1:nbins
        c = counts[k]
        if c > 0
            βrep = sumsβ[k] / c   # ビン内平均βを代表値に
            total += c * real(WignerD.wignerDjmn(l, m, n, 0.0, βrep, 0.0))
        end
    end

    return total / N
end

# ---------------------------
# 比較（誤差 + 時間）
# ---------------------------
"""
direct と binned を比較して、誤差と時間を返す。

返り値例:
(
  direct = ...,
  binned = ...,
  abs_err = ...,
  rel_err = ...,
  t_direct = ...,
  t_binned = ...,
  speedup = ...
)
"""
function compare_direct_vs_binned(
    l::Int, m::Int, n::Int, betas::AbstractVector{<:Real};
    nbins::Int = 2000,
    warmup::Bool = true,
)
    # JITウォームアップ（時間比較を少し公平に）
    if warmup && !isempty(betas)
        β0 = float(betas[1])
        _ = real(WignerD.wignerDjmn(l, m, n, 0.0, β0, 0.0))
        _ = mean_wignerd_binned(l, m, n, betas; nbins=nbins)
    end

    t1 = @elapsed direct = mean_wignerd_direct(l, m, n, betas)
    t2 = @elapsed binned = mean_wignerd_binned(l, m, n, betas; nbins=nbins)

    abs_err = abs(binned - direct)
    rel_err = abs_err / (abs(direct) + 1e-300)

    return (
        direct = direct,
        binned = binned,
        abs_err = abs_err,
        rel_err = rel_err,
        t_direct = t1,
        t_binned = t2,
        speedup = t1 / max(t2, 1e-12),
        nbins = nbins,
        Nbeta = length(betas),
    )
end

# ---------------------------
# nbins を変えてスキャン
# ---------------------------
function scan_nbins_compare(
    l::Int, m::Int, n::Int, betas::AbstractVector{<:Real},
    nbins_list::AbstractVector{Int};
    warmup::Bool = true,
)
    results = NamedTuple[]
    for nb in nbins_list
        push!(results, compare_direct_vs_binned(l, m, n, betas; nbins=nb, warmup=warmup))
        warmup = false  # 2回目以降は不要
    end
    return results
end

# ---------------------------
# 見やすく表示
# ---------------------------
function print_compare_results(results)
    println("nbins\tNbeta\trel_err\t\tabs_err\t\tt_direct[s]\tt_binned[s]\tspeedup")
    for r in results
        println("$(r.nbins)\t$(r.Nbeta)\t$(r.rel_err)\t$(r.abs_err)\t$(r.t_direct)\t$(r.t_binned)\t$(r.speedup)")
    end
end

print_compare_results (generic function with 1 method)

In [85]:
# 例: β をランダムに作る（0〜π）
N = 100_000
betas = rand(N) .* π

l, m, n = 100, 70, 20

# まず1点比較
r = compare_direct_vs_binned(l, m, n, betas; nbins=2000)
@show r

# nbins をスキャン
nbins_list = [100, 300, 1000, 3000, 10000]
results = scan_nbins_compare(l, m, n, betas, nbins_list)
print_compare_results(results)

r = (direct = 0.007779863472995329, binned = 0.007779898192622879, abs_err = 3.471962754970914e-8, rel_err = 4.462755377420745e-6, t_direct = 0.308981833, t_binned = 0.006044334, speedup = 51.11925201353863, nbins = 2000, Nbeta = 100000)
nbins	Nbeta	rel_err		abs_err		t_direct[s]	t_binned[s]	speedup
100	100000	0.0015619987849513112	1.2152137291905792e-5	0.30810125	0.00076575	402.3522690173033
300	100000	0.0006962490830034285	5.416722808964866e-6	0.307005833	0.001332625	230.37676240502768
1000	100000	9.612969123993582e-6	7.478758735478958e-8	0.307594625	0.00325375	94.53542066845947
3000	100000	2.7204841415556003e-6	2.116499520175147e-8	0.307322667	0.008958125	34.306583911253746
10000	100000	4.69517413374799e-7	3.652781374249847e-9	0.311760833	0.028638417	10.886105646132606
